# 🌱 Analyse Dataset Course_green - E-commerce Écologique

## Vue d'ensemble des données
Ce notebook analyse votre dataset **Course_green** contenant :
- **7 tables** : green_categories, green_product, green_price, green_pdt_segment, green_promo, green_sales, green_stock
- **4,693 lignes** au total réparties sur les différentes tables
- **Données e-commerce** avec focus sur les produits écologiques

### Structure découverte :
- **422 catégories** organisées en hiérarchie (category_1, category_2, category_3)
- **1,000 produits** avec segments, prix et promotions
- **244 transactions de vente** à analyser
- **27 promotions** actives ou passées

## 1. Installation et Import des Bibliothèques

Commençons par installer et importer toutes les bibliothèques nécessaires.

In [ ]:
# Configuration et imports pour l'analyse Course_green
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Configuration du style des graphiques
plt.style.use('default')
sns.set_palette("husl")

# Charger les variables d'environnement
load_dotenv()

# Configuration BigQuery pour Course_green
PROJECT_ID = "versatile-vine-462408-g0"
DATASET_ID = "Course_green"

print("🌱 Configuration Course_green:")
print(f"📊 Projet: {PROJECT_ID}")
print(f"📁 Dataset: {DATASET_ID}")

# Initialiser le client BigQuery
client = bigquery.Client(project=PROJECT_ID)

## 2. Configuration de l'Authentification BigQuery

Configurons l'authentification pour accéder à vos données BigQuery.

In [ ]:
# Explorer toutes les tables du dataset Course_green
def get_table_info():
    """Récupère les informations sur toutes les tables"""
    dataset_ref = client.dataset(DATASET_ID)
    tables = list(client.list_tables(dataset_ref))

    table_data = []
    for table in tables:
        table_ref = dataset_ref.table(table.table_id)
        table_obj = client.get_table(table_ref)

        table_data.append({
            'table_name': table.table_id,
            'rows': table_obj.num_rows,
            'columns': len(table_obj.schema),
            'size_mb': round(table_obj.num_bytes / (1024*1024), 2),
            'created': table_obj.created.strftime('%Y-%m-%d')
        })

    return pd.DataFrame(table_data)

# Afficher les informations des tables
table_info = get_table_info()
print("📊 TABLES COURSE_GREEN:")
print("="*50)
display(table_info)

## 3. Connexion à BigQuery

Créons une connexion client BigQuery et testons-la.

In [ ]:
# Import du client personnalisé
import sys
sys.path.append('../scripts')
from bigquery_client import BigQueryClient

# Créer le client BigQuery
try:
    bq_client = BigQueryClient()
    print("✅ Connexion BigQuery établie")

    # Lister les datasets disponibles
    datasets = bq_client.list_datasets()
    print(f"📊 Datasets disponibles: {datasets}")

except Exception as e:
    print(f"❌ Erreur de connexion: {e}")
    print("Vérifiez vos credentials et votre configuration")

## 4. Requête de Données depuis BigQuery

Écrivons des requêtes SQL pour extraire vos données.

In [ ]:
# Exemple de requête - MODIFIEZ SELON VOS DONNÉES
sample_query = f"""
SELECT
    *
FROM `{PROJECT_ID}.{DATASET_ID}.your_table_name`
LIMIT 1000
"""

# Requête pour explorer la structure de vos données
schema_query = f"""
SELECT
    column_name,
    data_type,
    is_nullable
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'your_table_name'
ORDER BY ordinal_position
"""

print("📝 Requêtes SQL prêtes")
print("Exemple de requête:")
print(sample_query)

## 5. Chargement des Données dans un DataFrame

Convertissons les résultats BigQuery en DataFrames pandas.

In [ ]:
# REMPLACEZ CETTE REQUÊTE PAR LA VÔTRE
your_query = """
-- Exemple de requête - remplacez par votre requête
SELECT
    date_column,
    category_column,
    value_column,
    COUNT(*) as count_records
FROM `your-project.your-dataset.your-table`
WHERE date_column >= '2024-01-01'
GROUP BY date_column, category_column, value_column
ORDER BY date_column DESC
LIMIT 10000
"""

# Exécution de la requête
try:
    print("🔄 Exécution de la requête...")
    df = bq_client.query_to_dataframe(your_query)

    if df is not None:
        print(f"✅ Données chargées: {df.shape[0]} lignes, {df.shape[1]} colonnes")
        print(f"📊 Aperçu des données:")
        display(df.head())
    else:
        print("❌ Aucune donnée récupérée")

except Exception as e:
    print(f"❌ Erreur lors de l'exécution: {e}")
    # Créer un DataFrame d'exemple pour continuer
    df = pd.DataFrame({
        'date': pd.date_range('2024-01-01', periods=100),
        'category': np.random.choice(['A', 'B', 'C'], 100),
        'value': np.random.randn(100) * 100 + 500
    })
    print("📊 Utilisation de données d'exemple pour la démonstration")

## 6. Exploration Basique des Données

Explorons la structure et les caractéristiques de nos données.

In [ ]:
# Informations générales sur le dataset
print("📋 INFORMATIONS GÉNÉRALES")
print(f"Forme des données: {df.shape}")
print(f"\n📊 TYPES DE DONNÉES:")
print(df.dtypes)
print(f"\n🔍 INFORMATIONS DÉTAILLÉES:")
df.info()
print(f"\n📈 STATISTIQUES DESCRIPTIVES:")
display(df.describe())

In [ ]:
# Vérification des valeurs manquantes
print("🔍 VALEURS MANQUANTES:")
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Valeurs manquantes': missing_data,
    'Pourcentage': missing_percent
})
display(missing_df[missing_df['Valeurs manquantes'] > 0])

# Valeurs uniques par colonne
print("\n🎯 CARDINALITÉ DES COLONNES:")
for col in df.columns:
    unique_count = df[col].nunique()
    print(f"{col}: {unique_count} valeurs uniques")

## 7. Nettoyage et Préparation des Données

Nettoyons et préparons les données pour l'analyse.

In [ ]:
# Créer une copie pour le nettoyage
df_clean = df.copy()

# Nettoyage basique
print("🧹 NETTOYAGE DES DONNÉES")

# Supprimer les doublons
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
duplicates_removed = initial_rows - len(df_clean)
print(f"Doublons supprimés: {duplicates_removed}")

# Traitement des valeurs manquantes (exemple)
numeric_columns = df_clean.select_dtypes(include=[np.number]).columns
categorical_columns = df_clean.select_dtypes(include=['object']).columns

# Remplir les valeurs manquantes numériques avec la médiane
for col in numeric_columns:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"Valeurs manquantes dans {col} remplacées par la médiane: {median_val}")

# Remplir les valeurs manquantes catégorielles avec le mode
for col in categorical_columns:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown'
        df_clean[col].fillna(mode_val, inplace=True)
        print(f"Valeurs manquantes dans {col} remplacées par: {mode_val}")

print(f"\n✅ Nettoyage terminé. Shape finale: {df_clean.shape}")

In [ ]:
# Détection et traitement des outliers (exemple pour colonnes numériques)
print("🎯 DÉTECTION DES OUTLIERS")

for col in numeric_columns:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]
    print(f"{col}: {len(outliers)} outliers détectés ({len(outliers)/len(df_clean)*100:.2f}%)")

    # Option: supprimer les outliers extrêmes
    # df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

## 8. Visualisation des Données

Créons des visualisations pour comprendre nos données.

In [ ]:
# Import du module de visualisation
from visualization_utils import DataVisualizer

# Créer l'instance du visualiseur
viz = DataVisualizer()

print("📊 CRÉATION DES VISUALISATIONS")

In [ ]:
# Graphiques de distribution pour les colonnes numériques
if len(numeric_columns) > 0:
    fig, axes = plt.subplots(len(numeric_columns), 2, figsize=(15, 5*len(numeric_columns)))
    if len(numeric_columns) == 1:
        axes = axes.reshape(1, -1)

    for i, col in enumerate(numeric_columns):
        # Histogramme
        axes[i, 0].hist(df_clean[col], bins=30, alpha=0.7)
        axes[i, 0].set_title(f'Distribution de {col}')
        axes[i, 0].set_xlabel(col)
        axes[i, 0].set_ylabel('Fréquence')

        # Box plot
        axes[i, 1].boxplot(df_clean[col])
        axes[i, 1].set_title(f'Box Plot de {col}')
        axes[i, 1].set_ylabel(col)

    plt.tight_layout()
    plt.show()
else:
    print("Aucune colonne numérique trouvée pour les distributions")

In [ ]:
# Graphiques interactifs avec Plotly
print("🎨 GRAPHIQUES INTERACTIFS PLOTLY")

# Exemple: graphique en barres des catégories (si applicable)
if len(categorical_columns) > 0:
    first_cat_col = categorical_columns[0]
    cat_counts = df_clean[first_cat_col].value_counts()

    fig_bar = px.bar(
        x=cat_counts.index,
        y=cat_counts.values,
        title=f'Distribution de {first_cat_col}',
        labels={'x': first_cat_col, 'y': 'Nombre'}
    )
    fig_bar.show()

    # Sauvegarder le graphique
    viz.save_plot(fig_bar, f'distribution_{first_cat_col}', 'html')

# Exemple: graphique de corrélation (si applicable)
if len(numeric_columns) >= 2:
    corr_matrix = df_clean[numeric_columns].corr()

    fig_heatmap = px.imshow(
        corr_matrix,
        title='Matrice de Corrélation',
        labels=dict(color="Corrélation"),
        color_continuous_scale='RdBu'
    )
    fig_heatmap.show()

    # Sauvegarder
    viz.save_plot(fig_heatmap, 'correlation_matrix', 'html')

In [ ]:
# Graphique temporel (si vous avez une colonne date)
date_columns = df_clean.select_dtypes(include=['datetime64']).columns

if len(date_columns) > 0 and len(numeric_columns) > 0:
    date_col = date_columns[0]
    value_col = numeric_columns[0]

    # Agrégation par date
    daily_data = df_clean.groupby(df_clean[date_col].dt.date)[value_col].mean().reset_index()
    daily_data.columns = ['date', 'average_value']

    fig_time = px.line(
        daily_data,
        x='date',
        y='average_value',
        title=f'Evolution temporelle de {value_col}'
    )
    fig_time.show()

    # Sauvegarder
    viz.save_plot(fig_time, 'evolution_temporelle', 'html')
else:
    print("Aucune colonne de date trouvée pour l'analyse temporelle")

## 9. Export des Résultats pour Affichage Web

Exportons les données et visualisations dans des formats adaptés au web.

In [ ]:
# Créer le dossier d'export s'il n'existe pas
export_dir = '../data'
os.makedirs(export_dir, exist_ok=True)

print("📤 EXPORT DES DONNÉES")

# 1. Export CSV des données nettoyées
csv_path = f'{export_dir}/cleaned_data_{datetime.now().strftime("%Y%m%d_%H%M")}.csv'
df_clean.to_csv(csv_path, index=False)
print(f"✅ Données exportées en CSV: {csv_path}")

# 2. Export JSON pour l'utilisation web
json_path = f'{export_dir}/data_summary_{datetime.now().strftime("%Y%m%d_%H%M")}.json'
summary_data = {
    'metadata': {
        'total_records': len(df_clean),
        'columns': list(df_clean.columns),
        'export_date': datetime.now().isoformat()
    },
    'statistics': df_clean.describe().to_dict() if len(numeric_columns) > 0 else {},
    'sample_data': df_clean.head(10).to_dict('records')
}

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, ensure_ascii=False, indent=2, default=str)
print(f"✅ Résumé exporté en JSON: {json_path}")

In [ ]:
# 3. Création d'un dashboard HTML complet
dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Dashboard Analytique</title>
    <meta charset="utf-8">
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        .dashboard-header {{ background-color: #f0f0f0; padding: 20px; border-radius: 10px; }}
        .metrics {{ display: flex; justify-content: space-around; margin: 20px 0; }}
        .metric {{ text-align: center; padding: 10px; background-color: #e6f3ff; border-radius: 5px; }}
        .chart-container {{ margin: 20px 0; padding: 15px; border: 1px solid #ddd; border-radius: 10px; }}
    </style>
</head>
<body>
    <div class="dashboard-header">
        <h1>Dashboard Analytique</h1>
        <p>Généré le: {datetime.now().strftime('%d/%m/%Y %H:%M')}</p>
        <p>Nombre total d'enregistrements: {len(df_clean):,}</p>
    </div>

    <div class="metrics">
        <div class="metric">
            <h3>{len(df_clean):,}</h3>
            <p>Enregistrements</p>
        </div>
        <div class="metric">
            <h3>{len(df_clean.columns)}</h3>
            <p>Colonnes</p>
        </div>
        <div class="metric">
            <h3>{df_clean.isnull().sum().sum()}</h3>
            <p>Valeurs manquantes</p>
        </div>
    </div>

    <div class="chart-container">
        <h2>Visualisations intégrées</h2>
        <p>Les graphiques Plotly peuvent être intégrés ici via iframe ou en copiant le code HTML généré.</p>
    </div>
</body>
</html>
"""

dashboard_path = f'{export_dir}/dashboard_{datetime.now().strftime("%Y%m%d_%H%M")}.html'
with open(dashboard_path, 'w', encoding='utf-8') as f:
    f.write(dashboard_html)
print(f"✅ Dashboard HTML créé: {dashboard_path}")

In [ ]:
# 4. Génération de code pour intégration Flask
flask_code = f"""
# Code d'exemple pour intégrer dans votre application Flask
from flask import Flask, render_template, jsonify
import pandas as pd
import json

app = Flask(__name__)

@app.route('/api/data')
def get_data():
    # Charger les données
    df = pd.read_csv('{csv_path}')
    return jsonify(df.to_dict('records'))

@app.route('/api/summary')
def get_summary():
    with open('{json_path}', 'r') as f:
        summary = json.load(f)
    return jsonify(summary)

@app.route('/dashboard')
def dashboard():
    return render_template('dashboard.html')

if __name__ == '__main__':
    app.run(debug=True)
"""

flask_path = f'{export_dir}/flask_integration_example.py'
with open(flask_path, 'w', encoding='utf-8') as f:
    f.write(flask_code)
print(f"✅ Exemple d'intégration Flask: {flask_path}")

print("\n🎉 EXPORT TERMINÉ!")
print(f"Tous les fichiers sont disponibles dans: {export_dir}")

## Résumé et Prochaines Étapes

### Ce que nous avons accompli:
1. ✅ Connexion à BigQuery
2. ✅ Extraction et nettoyage des données
3. ✅ Analyses exploratoires
4. ✅ Création de visualisations
5. ✅ Export pour intégration web

### Fichiers générés:
- **CSV**: Données nettoyées pour utilisation
- **JSON**: Résumé et métadonnées
- **HTML**: Dashboard et graphiques
- **Python**: Code d'exemple Flask

### Prochaines étapes:
1. Personnalisez les requêtes SQL selon vos données
2. Adaptez les visualisations à vos besoins
3. Intégrez les composants dans votre site web
4. Configurez l'actualisation automatique des données
5. Testez les performances et optimisez si nécessaire

### Intégration dans votre site:
- Utilisez les fichiers HTML générés comme iframes
- Intégrez les APIs JSON dans votre backend
- Copiez les graphiques Plotly dans vos templates
- Automatisez l'exécution de ce notebook pour des mises à jour régulières